# MOZYME GPU: perfil por secciones (CPU vs GPU-DIAGG vs SCF residente)

Clona `https://github.com/juvenalyosa/mopac_gpu`, compila MOPAC con CUDA y corre `scripts/mozyme_section_profile.py` sobre las moléculas de publicación.

Runtime: **GPU (A100 o H100 preferido; T4/L4 funcionan pero con FP64 lento)**.

Abrir directamente desde GitHub: `https://colab.research.google.com/github/juvenalyosa/mopac_gpu/blob/main/colab/mozyme_diagg_profile_colab.ipynb`


## 1. GPU

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)

## 2. Obtener el código fuente (git clone; opcionalmente un zip local)

In [ ]:
from pathlib import Path
import shutil, subprocess, zipfile

REPO_URL = 'https://github.com/juvenalyosa/mopac_gpu.git'
BRANCH = 'main'
USE_LOCAL_ZIP = False  # True: subir un zip creado con scripts/create_colab_gpu_zip.py en vez de clonar

CONTENT = Path('/content')
SRC = CONTENT / 'mopac_src'
BUILD = CONTENT / 'mopac_build'

if SRC.exists():
    shutil.rmtree(SRC)
if USE_LOCAL_ZIP:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.endswith('.zip'))
    SRC.mkdir(parents=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(SRC)
    entries = [p for p in SRC.iterdir() if not p.name.startswith('.')]
    if len(entries) == 1 and entries[0].is_dir() and (entries[0] / 'CMakeLists.txt').exists():
        SRC = entries[0]
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(SRC)], check=True)
    print(subprocess.run(['git', '-C', str(SRC), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert (SRC / 'CMakeLists.txt').exists(), f'CMakeLists.txt not found under {SRC}'
assert (SRC / 'scripts' / 'mozyme_section_profile.py').exists(), 'scripts/mozyme_section_profile.py missing'
print('source dir:', SRC)

## 3. Compilar MOPAC con GPU

In [ ]:
import subprocess, shutil

def run(cmd, **kw):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    return subprocess.run([str(c) for c in cmd], check=True, **kw)

# apt-get needs fresh package lists in a new Colab session (exit status 100 otherwise);
# retry once because the Ubuntu mirrors occasionally time out.
pkgs = ['cmake', 'gfortran', 'ninja-build', 'libblas-dev', 'liblapack-dev']
for attempt in range(2):
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    r = subprocess.run(['apt-get', 'install', '-y', '-qq', '--no-install-recommends', *pkgs],
                       check=False, capture_output=True, text=True)
    if r.returncode == 0:
        break
    print(r.stdout[-2000:], r.stderr[-2000:])
    if attempt == 1:
        raise SystemExit('apt-get install failed twice; run !apt-get update && !apt-get install ' + ' '.join(pkgs))
if BUILD.exists():
    shutil.rmtree(BUILD)
cmake_cmd = ['cmake', '-S', SRC, '-B', BUILD, '-GNinja', '-DGPU=ON', '-DTESTS=OFF', '-DCMAKE_BUILD_TYPE=RelWithDebInfo']
try:
    run(cmake_cmd + ['-DCUDA_ARCHS=native'])
except subprocess.CalledProcessError:
    shutil.rmtree(BUILD, ignore_errors=True)
    run(cmake_cmd + ['-DCUDA_ARCHS=all'])
run(['cmake', '--build', BUILD, '--target', 'mopac', '--parallel', '2'])
MOPAC = BUILD / 'mopac'
assert MOPAC.exists(), 'mopac executable not built'
print('OK:', MOPAC)

## 4. Perfil por secciones

Modos: `cpu` (referencia), `gpu-diagg` (loop SCF en CPU con sólo DIAGG en los kernels paralelos nuevos), `resident` (SCF completo residente en GPU, modo estricto). Empieza con crambina (~30 s por modo en CPU); ubiquitina tarda ~40 s por modo.

## 4b. Depuración de los kernels DIAGG (crambina, modo gpu-diagg)

Corre sólo crambina en `gpu-diagg` con `MOPAC_MOZYME_DIAGG_DEBUG=1` (sincroniza tras cada kernel e imprime `[DIAGG DEBUG] ...`), y luego bajo `compute-sanitizer` (memcheck) para detectar accesos inválidos en device y bajo `--tool racecheck` para carreras en memoria compartida. Muestra las últimas líneas de cada corrida.

In [ ]:
import os, shutil, subprocess
RUN_RACECHECK = False  # True: add the compute-sanitizer racecheck pass (slow)

DBG = CONTENT / 'diagg_debug'
shutil.rmtree(DBG, ignore_errors=True)
deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn.mop'

def run_debug(tag, prefix, mode='gpu-diagg', head=0, deck=deck, debug='1'):
    env = dict(os.environ, MOPAC_MOZYME_DIAGG_DEBUG=debug)
    cmd = [*prefix, sys.executable, str(SRC / 'scripts/mozyme_section_profile.py'), str(MOPAC), str(deck),
           '--modes', mode, '--out-dir', str(DBG / tag), '--timeout', '1800']
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True, env=env)
    print(proc.stdout[-6000:])
    log = next(iter((DBG / tag).rglob('combined.log')), None)
    if log:
        lines = log.read_text(errors='ignore').splitlines()
        import re
        pat = re.compile(r'DIAGG DEBUG|GPU ERROR|Backtrace|\.cu:\d|\.F90:\d|^=+|Invalid|Error|error|FINAL HEAT|MOZYME GPU diagg|CYCLE:|resident_publish|resident_upload|MOZYME GPU SCF\] status')
        keep = [l for l in lines if pat.search(l)]
        if head:
            print('\n'.join(keep[:head]))
            print('   ...')
        print('\n'.join(keep[-60:]))
        (CONTENT / f'diagg_debug_{tag}.log').write_text('\n'.join(lines))

run_debug('plain', [])
run_debug('resident_debug', [], mode='resident', head=60)
# Hand-back regression (1SCF, ITRY=70): DENOUT=5 makes the resident SCF hand back after 4 iterations;
# the CPU then finishes that SCF from the published device state (the driver blocks the GPU for the
# rest of the SCF).  Expect one "status=fallback_cpu", no [GPU ERROR], heat -2901.681 within 0.01.
# A diverging CPU continuation shows as ITRY exhausted (~70 iterations, wrong or missing heat).
run_debug('denout_handback', [], mode='resident', debug='0',
          deck=SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_denout.mop')
san = shutil.which('compute-sanitizer') or '/usr/local/cuda/bin/compute-sanitizer'
if Path(san).exists():
    run_debug('sanitizer', [san, '--target-processes', 'all', '--print-limit', '20'])
    # racecheck: shared-memory races (diagg2 block kernel shares its staged lists across 8 warps).
    # 10-50x slower than memcheck (10-20 min on crambin): off by default, enable after touching shared memory.
    if RUN_RACECHECK:
        run_debug('racecheck', [san, '--tool', 'racecheck', '--target-processes', 'all', '--print-limit', '20'])
else:
    print('compute-sanitizer not found; skipping')


## 4c. Optimización de geometría (crambina, 3 ciclos): CPU vs residente

Mide pasos de optimización completos (SCF warm-start + gradientes). Los gradientes MOZYME hoy corren en CPU (`dcart_gradient` en la tabla).

El modo `resident-gradcheck` evalúa el gradiente MOZYME en CPU y en GPU en cada ciclo y escribe `[MOZYME GPU gradient] check ... max_abs_diff= rms_diff=` (se conserva el resultado CPU); `resident` usa directamente el gradiente GPU.


In [ ]:
import subprocess
opt_deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_opt.mop'
OPT_OUT = CONTENT / 'mozyme_opt_profile'
cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC), str(opt_deck),
       '--modes', 'cpu,resident-gradcheck,resident', '--out-dir', str(OPT_OUT), '--timeout', '3600']
print('$', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
import re
pat = re.compile(r'MOZYME tidy|MOZYME GPU SCF\] status|MOZYME GPU SCF\] resident_(upload|publish)|MOZYME GPU gradient|MOZYME GPU hcore|MOZYME GPU disp|MOZYME GPU hbond|strict_abort|CYCLE:|HEAT OF FORMATION|GRADIENT NORM|GPU ERROR|Backtrace|\.F90:\d')
for log in sorted(OPT_OUT.rglob('combined.log')):
    print('=====', log.relative_to(OPT_OUT))
    lines = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
    print('\n'.join(lines[:60]))


In [ ]:
import os, subprocess

MOLECULES = [
    'benchmarks/publication_inputs/mop/protein_crambin_1crn.mop',
    'benchmarks/publication_inputs/mop/protein_ubiquitin_1ubq.mop',
]
MODES = 'cpu,resident,default'
OUT = CONTENT / 'mozyme_section_profile'

cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC),
       *[str(SRC / m) for m in MOLECULES], '--modes', MODES, '--out-dir', str(OUT), '--timeout', '3600']
print('$', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
(CONTENT / 'mozyme_section_profile_summary.txt').write_text(proc.stdout + '\n' + proc.stderr)

## 4c-full. Optimización de crambina hasta convergencia: CPU vs resident

Hasta ahora todas las optimizaciones fueron de 3 ciclos sin converger, y el heat del ciclo 3 varía ~0.8 kcal/mol por el
orden no determinista de diagg2. Aquí se deja converger (hasta 100 ciclos, GNORM por defecto): los heats finales deben
coincidir dentro de 0.05 kcal/mol y la tabla del último paso da el coste real por paso caliente. CPU ~10 min.


In [ ]:
import subprocess, re
deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_optfull.mop'
OUT_F = CONTENT / 'mozyme_optfull_profile'
cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC), str(deck),
       '--modes', 'cpu,resident', '--out-dir', str(OUT_F), '--timeout', '7200']
print('$', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
out = proc.stdout.splitlines()
print('\n'.join(l for l in out if 'wall=' in l))
start = next((i for i, l in enumerate(out) if l.startswith('last geometry step')), None)
if start is not None:
    print('\n'.join(out[start:start + 30]))
pat = re.compile(r'CYCLE:|HEAT OF FORMATION|GRADIENT NORM|GPU ERROR|fallback_cpu reason|strict_abort')
for log in sorted(OUT_F.rglob('combined.log')):
    lines = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
    cycles = [l for l in lines if 'CYCLE:' in l]
    print('=====', log.relative_to(OUT_F), f'({len(cycles)} cycles)')
    print('\n'.join(cycles[:3] + ['   ...'] + cycles[-3:] + [l for l in lines if 'CYCLE:' not in l][-6:]))

# Energy check at a common geometry.  A run that exhausts CYCLES writes no .arc, only the restart
# file (.res) holding the optimizer's current geometry: RESTART 1SCF re-evaluates that geometry.
# Optimizer trajectories diverge (nondeterministic diagg2 order), so only heats at the same geometry
# are comparable: |dHf(resident - cpu)| must be < 0.05 kcal/mol.
import shutil
res = next(iter((OUT_F / deck.stem / 'resident').glob('*.res')), None)
if res is None:
    print('no .res from the resident run')
else:
    CHK = CONTENT / 'mozyme_optfull_check'
    shutil.rmtree(CHK, ignore_errors=True)
    CHK.mkdir(parents=True)
    shutil.copy(res, CHK / 'crambin_gpu_final_1scf.res')
    shutil.copy(SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_hydrogenated.pdb', CHK)
    (CHK / 'crambin_gpu_final_1scf.mop').write_text(
        'PM7 GEO_DAT="protein_crambin_1crn_hydrogenated.pdb" 1SCF RESTART MOZYME MOZYME_GPU MOZYME_MINBLK=16 '
        'PULAY SHIFT=-50 ITRY=200 GEO-OK NOCOMMENTS\n'
        '1SCF at the current geometry of the resident optimization (restart file): CPU vs resident heats\n\n')
    cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC),
           str(CHK / 'crambin_gpu_final_1scf.mop'), '--modes', 'cpu,resident', '--out-dir', str(CHK / 'out'),
           '--timeout', '1800']
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    print('\n'.join(l for l in proc.stdout.splitlines() if 'wall=' in l or 'dHf' in l))


## 4e. Hand-back: el SCF en CPU tras devolver el control (regresión)

Con `DENOUT=5` el SCF residente devuelve el control a la CPU tras 4 iteraciones. Diagnóstico del 18-09-2026: con los
ayudantes GPU antiguos del bucle CPU (diagg1 aocc/avir, density batch, eimp, cnvgz, helecz, rotprep) la CPU no convergía
(ITRY agotado) y con solo `diagg1_avir` apagado convergía a -2899.57 en vez de -2901.68. Desde entonces esos ayudantes son
opt-in. Aquí la corrida por defecto debe converger a -2901.68 (dentro de 0.01) en ~10 s; `legacy_helpers_on` reproduce el fallo
(~2 min) y `cpu_reference` da la referencia.


In [ ]:
import subprocess, re
DIAG = CONTENT / 'handback_diag'
deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn_denout.mop'
VARIANTS = [
    # (tag, mode, overrides)  -- overrides applied after the mode; KEY= unsets
    ('default_handback',    'resident',  ['MOPAC_GPU_VERBOSE=1']),
    ('legacy_helpers_on',   'resident',  ['MOPAC_GPU_VERBOSE=1', 'MOPAC_MOZYME_DIAGG1_AOCC_GPU=1',
                                          'MOPAC_MOZYME_DIAGG1_AVIR_GPU=1', 'MOPAC_MOZYME_DENSITY_BATCH_GPU=1',
                                          'MOPAC_MOZYME_EIMP_GPU=1', 'MOPAC_MOZYME_CNVGZ_GPU=1',
                                          'MOPAC_MOZYME_HELECZ_GPU=1', 'MOPAC_MOZYME_DIAGG2_ROTPREP_GPU=1']),
    ('cpu_reference',       'cpu',       []),
]
pat = re.compile(r'MOZYME GPU SCF\] status|MOZYME GPU SCF\] resident_(upload|publish)|GPU ERROR|FINAL HEAT|ITERATIONS|SCF CALCULATION FAILED|strict_abort')
for tag, mode, sets in VARIANTS:
    cmd = [sys.executable, str(SRC / 'scripts/mozyme_section_profile.py'), str(MOPAC), str(deck),
           '--modes', mode, '--out-dir', str(DIAG), '--timeout', '900', '--tag', '_' + tag]
    for item in sets:
        cmd += ['--set', item]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    out = proc.stdout.splitlines()
    summary = [l for l in out if 'wall=' in l or 'dHf' in l or 'fallback_cpu=' in l]
    print('\n'.join(summary))
    if tag == 'legacy_helpers_on':   # section table: which CPU-loop section is slow (diagg 1.8 s/call vs 0.5 on CPU)
        start = next((i for i, l in enumerate(out) if l.startswith('section')), None)
        if start is not None:
            print('\n'.join(out[start:start + 22]))
    for log in sorted((DIAG / deck.stem / (mode + '_' + tag)).rglob('combined.log')):
        keep = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
        print('\n'.join(keep[-12:]))
    print()


## 4d. Sistemas grandes: DNA (1BNA), tRNA (1EHZ), adenilato quinasa (1AKE, ~7000 átomos) y 1AKE sin ligando (apo, 6689 átomos)

Single points en modo `resident` y `default` (y `cpu` sólo para 1BNA/1EHZ; para 1AKE la CPU tarda ~1 h — active `RUN_CPU_1AKE` si quiere la referencia). Los calores de referencia CPU quedan en el `.out` de la corrida `cpu`.


In [ ]:
import subprocess
RUN_CPU_1AKE = False     # CPU reference for 1AKE apo (~10 min on Colab CPU; Mac reference: -30137.44797 kcal/mol)
RUN_FULL_1AKE = False    # 1AKE with the AP5 ligand: does not converge on CPU either (~70 min); off by default
RUN_TRNA = False         # tRNA 1EHZ: does not converge on CPU either (~20 min per mode); off by default
LARGE = [
    ('benchmarks/publication_inputs/mop/dna_dodecamer_1bna.mop', 'cpu,resident,default'),
    # protein only (the AP5 ligand carries alternate conformations that hurt SCF convergence on CPU and GPU)
    ('benchmarks/publication_inputs/mop/protein_adenylate_kinase_1ake_apo.mop',
     'cpu,resident,resident-noindex,default' if RUN_CPU_1AKE else 'resident,resident-noindex,default'),
]
if RUN_TRNA:
    LARGE.append(('benchmarks/publication_inputs/mop/rna_trna_1ehz.mop', 'cpu,resident,default'))
if RUN_FULL_1AKE:
    LARGE.append(('benchmarks/publication_inputs/mop/protein_adenylate_kinase_1ake.mop', 'resident,default'))
# 3-cycle geometry optimization of 1AKE apo: the "last geometry step" table is the warm-start
# per-step cost (SCF from the previous LMOs + hcore + gradient) that optimization/MD pays.
LARGE.append(('benchmarks/publication_inputs/mop/protein_adenylate_kinase_1ake_apo_opt.mop', 'resident'))
OUT_L = CONTENT / 'mozyme_large_profile'
for deck, modes in LARGE:
    cmd = [sys.executable, str(SRC / 'scripts' / 'mozyme_section_profile.py'), str(MOPAC), str(SRC / deck),
           '--modes', modes, '--out-dir', str(OUT_L), '--timeout', '7200']
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    print(proc.stdout)
    print(proc.stderr[-3000:])
    # host<->device transfer profile (bytes, ms, GB/s) and SCF status of each run of this deck
    import re
    pat = re.compile(r'MOZYME tidy|MOZYME GPU SCF\] resident_(upload|publish)|MOZYME GPU SCF\] status|GPU ERROR|CYCLE:')
    for log in sorted((OUT_L / Path(deck).stem).rglob('combined.log')):
        keep = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
        print('=====', log.relative_to(OUT_L))
        print('\n'.join(keep[-24:]))
(CONTENT / 'mozyme_large_profile_summary.txt').write_text('done')


## 4f. Nsight Compute: motivos de estancamiento de los kernels dominantes (opcional)

Perfila con `ncu` los tres kernels que dominan la iteración SCF a 7000 átomos (1AKE apo, 1SCF): `mozyme_diagg1_virtual_kernel` (count/fill),
`mozyme_density_pairs_kernel` y `mozyme_diagg2_block_kernel`. Salta los primeros lanzamientos para capturar iteraciones con los LMO a tamaño completo
(diagg1_virtual se lanza dos veces por iteración: 4 lanzamientos = una iteración par y una impar). Las secciones `WarpStateStats` (stall reasons),
`Occupancy` y `MemoryWorkloadAnalysis` distinguen latencia de cargas aleatorias de saturación de memoria compartida. Cada kernel se reproduce varias
veces (replay), ~2-4 min en total. Si `ncu` falla por permisos de contadores (ERR_NVGPUCTRPERM) o se atasca, prueba `NCU_REPLAY = 'application'`.


In [ ]:
import os, shutil, subprocess, re
RUN_NCU = True
NCU_REPLAY = 'kernel'   # 'application' if kernel replay misbehaves with the resident loop
NCU_DECK = SRC / 'benchmarks/publication_inputs/mop/protein_adenylate_kinase_1ake_apo.mop'
KERNELS = [  # (regex, launch-skip, launch-count, deck)
    ('mozyme_diagg1_virtual_kernel', 60, 4, 'protein_adenylate_kinase_1ake_apo.mop'),
    ('mozyme_density_pairs_kernel', 30, 2, 'protein_adenylate_kinase_1ake_apo.mop'),
    ('mozyme_diagg2_block_kernel', 30, 2, 'protein_adenylate_kinase_1ake_apo.mop'),
    # gradient kernels (one launch per geometry step of the 3-cycle optimization): the
    # finite-difference sp pair kernel, the d-pair probe kernel and the point-charge kernel
    ('mozyme_pair_gradient_kernel', 1, 1, 'protein_adenylate_kinase_1ake_apo_opt.mop'),
    ('mozyme_pair_energy_d_kernel', 1, 1, 'protein_adenylate_kinase_1ake_apo_opt.mop'),
    ('mozyme_point_gradient_kernel', 1, 1, 'protein_adenylate_kinase_1ake_apo_opt.mop'),
    ('mozyme_hcore_pairs_kernel', 1, 1, 'protein_adenylate_kinase_1ake_apo_opt.mop'),
]
DECKS = {k[3] for k in KERNELS}
SECTIONS = ['SpeedOfLight', 'LaunchStats', 'Occupancy', 'SchedulerStats', 'WarpStateStats', 'MemoryWorkloadAnalysis']
ncu = shutil.which('ncu') or '/usr/local/cuda/bin/ncu'
if RUN_NCU and Path(ncu).exists():
    NCU_DIR = CONTENT / 'ncu_profile'
    shutil.rmtree(NCU_DIR, ignore_errors=True)
    NCU_DIR.mkdir(parents=True)
    for deck_name in DECKS:
        deck_path = NCU_DECK.parent / deck_name
        shutil.copy(deck_path, NCU_DIR / deck_name)
        # the deck reads its geometry from GEO_DAT="...pdb" next to it: copy every quoted file too
        for ref in re.findall(r'"([^"]+)"', deck_path.read_text(errors='ignore')):
            src_ref = deck_path.parent / Path(ref).name
            if src_ref.exists():
                shutil.copy(src_ref, NCU_DIR / src_ref.name)
    print(subprocess.run([ncu, '--version'], capture_output=True, text=True).stdout.strip().splitlines()[-1])
    for name, skip, count, deck_name in KERNELS:
        cmd = [ncu, '--kernel-name', f'regex:{name}', '--launch-skip', str(skip), '--launch-count', str(count),
               '--replay-mode', NCU_REPLAY, '--print-summary', 'per-kernel']
        for s in SECTIONS:
            cmd += ['--section', s]
        cmd += [str(MOPAC), deck_name]
        print('$', ' '.join(cmd), flush=True)
        proc = subprocess.run(cmd, cwd=NCU_DIR, capture_output=True, text=True, timeout=1800)
        out = proc.stdout
        (NCU_DIR / f'{name}.txt').write_text(out + '\n--- stderr ---\n' + proc.stderr)
        # keep the profile report (from the first section header on), drop MOPAC's own output
        start = out.find('==PROF==')
        report = out[start:] if start >= 0 else out
        keep = [l for l in report.splitlines() if not l.startswith('==PROF==') or 'Profiling' in l]
        print('\n'.join(keep[-260:]))
        if proc.returncode != 0:
            print('rc =', proc.returncode)
            print(proc.stderr[-2000:])
        out_file = NCU_DIR / (Path(deck_name).stem + '.out')
        if out_file.exists():
            lines = out_file.read_text(errors='ignore').splitlines()
            gpu = [l for l in lines if 'MOZYME GPU SCF] status' in l or 'HEAT OF FORMATION' in l]
            print('\n'.join(gpu[-3:] + lines[-3:]))
            out_file.unlink()
else:
    print('ncu not found or RUN_NCU is False; skipping')

## 6. Test de aceptación (Fase 6): tolerancia CPU/GPU, gradiente, hand-back y NOGPU

`tests/check_mozyme_gpu_tolerance.py` es el test CTest `mozyme-gpu-tolerance` (el build de Colab no configura ctest,
así que se llama directamente). Crambina 1SCF y optimización de 3 ciclos: |dHf| <= 0.05 kcal/mol, rms del gradiente
<= 1e-3 kcal/mol/A, hand-back con DENOUT=5 convergiendo en CPU, y NOGPU sin ningún ayudante GPU. ~3 minutos.


In [ ]:
import subprocess
TOL = CONTENT / 'mozyme_gpu_tolerance'
mop = SRC / 'benchmarks/publication_inputs/mop'
cmd = [sys.executable, str(SRC / 'tests/check_mozyme_gpu_tolerance.py'), str(MOPAC),
       str(mop / 'protein_crambin_1crn.mop'), str(mop / 'protein_crambin_1crn_opt.mop'),
       '--handback', str(mop / 'protein_crambin_1crn_denout.mop'), '--work-dir', str(TOL)]
print('$', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
print('exit status', proc.returncode)


## 7. Benchmark contra tiempos publicados

**Correr la celda 8 (validación de métodos) antes de esta.**

La página de openmopac.net "Use of MKL and Multi-Threading to reduce computation time" publica tiempos de 1SCF del
solver convencional (diagonalización, sin MOZYME) de MOPAC2016 en un Mac Pro 2 × Xeon 2.93 GHz de 6 núcleos (2010):
crambina 642 átomos 468 s (12 s con MKL y 12 hilos), 1G6X 1455 átomos 8612 s (142), 1EZG 2064 átomos 22959 s (300),
1RNB 2066 átomos 34372 s (411), bacteriorrodopsina 3352 átomos 141773 s (1394). Esta celda corre las mismas proteínas
(PDB hidrogenados con ADD-H; para bacteriorrodopsina la página no da código PDB y se usa 1C3W) con MOZYME en CPU
(un núcleo, `MOPAC_NOGPU=1`) y en GPU con este binario, y escribe la tabla con las columnas publicadas al lado.
Los tiempos publicados son de otro solver y otra máquina: indican lo que esperaba un usuario de MOPAC2016 por el
mismo punto simple, no una aceleración a igual hardware.

Incluye además dos cúmulos de agua generados en el momento (`scripts/make_water_cluster.py`): 1000 H2O y 7052 H2O
(42,312 orbitales, 21,156 átomos), el tamaño del sistema mayor de Maia, Cabral y Rocha, J Mol Model 26, 313 (2020),
que reportan hasta 40× (SP2 en GPU, una NVIDIA K40) frente a un hilo de CPU. Ellos reemplazan las rotaciones LMO de
MOZYME por una purificación de matriz densidad; aquí se conserva el algoritmo LMO. La referencia CPU del cúmulo grande
y de 1AKE tarda 30 a 40 min cada una en Colab: por defecto solo GPU (`RUN_CPU_LARGE = True` para medirlas).


In [ ]:
import subprocess
RUN_CPU_LARGE = False   # CPU reference also for 1AKE apo (6689 atoms) and the 7052-water cluster (~30-40 min each)
PUB = CONTENT / 'published_benchmark'
cmd = [sys.executable, str(SRC / 'scripts/published_benchmark.py'), str(MOPAC), '--inputs-dir',
       str(SRC / 'benchmarks/publication_inputs/mop'), '--out-dir', str(PUB), '--modes', 'cpu,gpu',
       '--skip-cpu-above', '0' if RUN_CPU_LARGE else '5000']
print('$', ' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr[-3000:])


## 7b. 1G6X (BPTI con sulfatos): comprobación tras corregir la entrada

Causa encontrada (2026-09-25): los sulfatos SO4 A 64 y 65 están en posiciones especiales del cristal
(`REMARK 375`, ocupación 0.5); el PDB solo trae S, O1 y O3 y el resto es una copia de simetría. MOZYME los convertía
en fragmentos "SO2(2−)" (de ahí la carga −4 y la energía inicial de +22616 kcal/mol) y el SCF quedaba mal condicionado:
112 iteraciones en CPU y energías distintas en cada corrida GPU. `prepare_publication_benchmark_inputs.py` ahora
descarta esos grupos y MOPAC comprueba la entrada (`INPUT CHEMISTRY CHECK`: grupos incompletos, posiciones especiales,
cargas formales imposibles; los errores detienen el trabajo salvo `LET`). Con la entrada corregida la CPU converge en
29 iteraciones. Esta celda repite CPU, `resident` y `default` y la GPU dos veces más: ΔHf debe quedar por debajo de
0.05 y la dispersión entre corridas GPU en el nivel habitual (< 0.01).

In [ ]:
import subprocess
OUT_D = CONTENT / 'diag_1g6x'
# second deck: the same protein without the seven (ADD-H protonated) sulfates and the ethylene glycol
RUNS = [('protein_1g6x.mop', 'cpu,resident,default', ''), ('protein_1g6x.mop', 'default', '_rep2'),
        ('protein_1g6x.mop', 'default', '_rep3'),
        ('protein_1g6x_noions.mop', 'cpu,default', ''), ('protein_1g6x_noions.mop', 'default', '_rep2'),
        ('protein_1g6x_noions.mop', 'default', '_rep3')]
for deck_name, modes, tag in RUNS:
    deck = SRC / 'benchmarks/publication_inputs/mop' / deck_name
    cmd = [sys.executable, str(SRC / 'scripts/mozyme_section_profile.py'), str(MOPAC), str(deck),
           '--modes', modes, '--out-dir', str(OUT_D), '--timeout', '1800'] + (['--tag', tag] if tag else [])
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    out = proc.stdout
    head = out.find('section ')
    print(out[:head] if head > 0 else out[-4000:])
    for line in out.splitlines():
        if 'last resident run' in line or 'dHf' in line or 'iterations=' in line:
            print(line)
    print(proc.stderr[-2000:])
for log in sorted(OUT_D.rglob('combined.log')):
    for l in log.read_text(errors='ignore').splitlines():
        if 'LMO orthogonality' in l or 'energy_recheck' in l:
            print(f'    {log.parent.parent.name}/{log.parent.name}: {l.strip()}')

## 7c. 1G6X con criterio de convergencia estricto (opcional)

Celda de diagnóstico de la versión anterior de la entrada (con los medios sulfatos, la CPU nunca cumplía
`SCFCRT=1.D-4` en 300 iteraciones). Con la entrada corregida debe converger en CPU y GPU; se deja para confirmarlo.
Imprime para cada corrida la línea `[MOZYME LMO orthogonality]` y `energy_recheck`.

In [ ]:
import subprocess, shutil, re
T7C = CONTENT / 'diag_1g6x_tight'
T7C.mkdir(parents=True, exist_ok=True)
src_deck = SRC / 'benchmarks/publication_inputs/mop/protein_1g6x.mop'
lines = src_deck.read_text().splitlines()
lines[0] = lines[0].replace(' 1SCF ', ' 1SCF SCFCRT=1.D-4 ').replace('ITRY=200', 'ITRY=300')
deck = T7C / 'protein_1g6x_tight.mop'
deck.write_text('\n'.join(lines) + '\n')
for ref in re.findall(r'"([^"]+)"', lines[0]):
    shutil.copy(src_deck.parent / Path(ref).name, T7C / Path(ref).name)
def show_orthogonality(out_dir):
    for log in sorted(Path(out_dir).rglob('combined.log')):
        for l in log.read_text(errors='ignore').splitlines():
            if 'LMO orthogonality' in l or 'energy_recheck' in l:
                print(f'    {log.parent.name}: {l.strip()}')
for modes, tag in [('cpu,default', ''), ('default', '_rep2'), ('default', '_rep3')]:
    cmd = [sys.executable, str(SRC / 'scripts/mozyme_section_profile.py'), str(MOPAC), str(deck),
           '--modes', modes, '--out-dir', str(T7C / 'out'), '--timeout', '3600'] + (['--tag', tag] if tag else [])
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    out = proc.stdout
    head = out.find('section ')
    print(out[:head] if head > 0 else out[-3000:])
    for line in out.splitlines():
        if 'last resident run' in line or 'dHf' in line:
            print(line)
    print(proc.stderr[-1500:])
show_orthogonality(T7C / 'out')


## 8. Métodos semiempíricos en la ruta GPU (validación)

Crambina y el dodecámero de DNA (fósforo: orbitales d en PM6/PM7) en 1SCF con MNDO, AM1, PM3, RM1, PM6, PM6-D3H4 y PM7:
CPU (`MOPAC_NOGPU=1`) frente a GPU (valores por defecto). Pasa si |ΔHf| ≤ 0.05 kcal/mol, el SCF residente reporta
`success` y no hay líneas de error GPU. En crambina corre además la optimización de 3 ciclos en GPU con los modos de
comprobación de hcore, gradiente y dispersión/puentes de H (`MOPAC_GPU_HCORE_CHECK`, `MOPAC_GPU_GRAD_CHECK`,
`MOPAC_GPU_DISP_CHECK`) e imprime las diferencias reportadas por método.


In [ ]:
import subprocess
MET = CONTENT / 'mozyme_gpu_methods'
mop = SRC / 'benchmarks/publication_inputs/mop'
for deck, extra in [('protein_crambin_1crn.mop', ['--opt', '--opt-deck', str(mop / 'protein_crambin_1crn_opt.mop')]),
                    ('dna_dodecamer_1bna.mop', [])]:
    cmd = [sys.executable, str(SRC / 'scripts/check_mozyme_gpu_methods.py'), str(MOPAC),
           '--work-dir', str(MET / Path(deck).stem), '--deck', str(mop / deck)] + extra
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    print(proc.stdout)
    print(proc.stderr[-3000:])


## 8b. Diagnóstico de PM3 y PM6 en crambina (etapa de check/tidy del dispositivo)

Con PM3 y PM6 el SCF residente devolvió el control con `reason=backend_missing_stages stage_missing=1022`: falló la
primera etapa después de la subida (tidy + check en el dispositivo). Esta celda corre esos decks en modo `default` con
`MOPAC_MOZYME_DIAGG_DEBUG=1` e imprime los volcados de tidy y check del dispositivo (índice del primer LMO malo, error
de normalización acumulado, resultados de tidy) junto con las líneas de estado.


In [ ]:
import subprocess, re
DIAG8 = CONTENT / 'diag_methods'
DIAG8.mkdir(parents=True, exist_ok=True)
src_deck = SRC / 'benchmarks/publication_inputs/mop/protein_crambin_1crn.mop'
for method in ['PM3', 'PM6']:
    mdir = DIAG8 / method
    mdir.mkdir(parents=True, exist_ok=True)
    lines = src_deck.read_text().splitlines()
    lines[0] = method + lines[0][3:]
    deck = mdir / src_deck.name
    deck.write_text('\n'.join(lines) + '\n')
    for ref in re.findall(r'"([^"]+)"', lines[0]):
        shutil.copy(src_deck.parent / Path(ref).name, mdir / Path(ref).name)
    cmd = [sys.executable, str(SRC / 'scripts/mozyme_section_profile.py'), str(MOPAC), str(deck),
           '--modes', 'default', '--out-dir', str(mdir / 'out'), '--timeout', '1800',
           '--set', 'MOPAC_MOZYME_DIAGG_DEBUG=1']
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=SRC, capture_output=True, text=True)
    print(proc.stdout[:1500])
    log = next(iter((mdir / 'out').rglob('combined.log')), None)
    if log:
        pat = re.compile(r'DIAGG DEBUG\] resident (tidy|check)|MOZYME GPU SCF\] status|stage_missing|GPU ERROR|FINAL HEAT')
        keep = [l for l in log.read_text(errors='ignore').splitlines() if pat.search(l)]
        print('\n'.join(keep[:40]))
        print('   ...')
        print('\n'.join(keep[-12:]))
    print(proc.stderr[-1500:])


## 5. Marcadores GPU y descarga

Muestra las líneas `[MOZYME GPU ...]` relevantes de cada corrida (éxito, fallback, abort) y empaqueta los logs para descargar.

In [ ]:
import re, shutil
from google.colab import files

pattern = re.compile(r'\[MOZYME GPU (diagg1_construct|diagg2_rotate|SCF)\]|strict_abort|fallback_cpu|MOZYME_RESIDENT_STAGE|GPU ERROR|CUDA error|cuda_context|Fortran runtime error|Segmentation|Program received|free\(\)|malloc|Backtrace|FINAL HEAT|SCF CALCULATION FAILED')
for log in sorted(OUT.rglob('combined.log')):
    print('=====', log.relative_to(OUT))
    lines = [l for l in log.read_text(errors='ignore').splitlines() if pattern.search(l)]
    for l in lines[:40]:
        print('  ', l[:200])
    if len(lines) > 40:
        print(f'   ... {len(lines) - 40} more')

archive = shutil.make_archive(str(CONTENT / 'mozyme_section_profile_logs'), 'zip', OUT)
files.download(archive)
files.download(str(CONTENT / 'mozyme_section_profile_summary.txt'))